# Risk Model: XGBoost + Calibration

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn import metrics

sys.path.insert(0,'..')
from src.preprocessing import load_raw, clean, get_model_arrays
from src.risk_model import train_risk_model, evaluate_risk_model, compute_baseline_targeting

In [ ]:
df = clean(load_raw())
X, y, treatment = get_model_arrays(df)
print(f"Dataset: {X.shape}, Readmission rate: {y.mean():.2%}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
print("Training risk model...")
model = train_risk_model(X_train, y_train)
print("Done.")

In [ ]:
metrics = evaluate_risk_model(model, X_test, y_test)
print("Test set metrics:")
[print(f"  {k}: {v:.4f}") for k,v in metrics.items()]

In [ ]:
baseline = compute_baseline_targeting(model, X_test, y_test)
print("\nBaseline risk targeting — readmissions captured:")
[print(f"  Top {int(k*100)}%: {v:.1%} of all readmissions") for k,v in baseline.items()]

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, CalibrationDisplay
fig, axes = plt.subplots(1,2,figsize=(12,5))
y_prob = model.predict_proba(X_test)[:,1]
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0], name='XGBoost+Cal')
axes[0].set_title(f'ROC Curve (AUC={metrics["auc"]:.3f})')
CalibrationDisplay.from_predictions(y_test, y_prob, n_bins=10, ax=axes[1])
axes[1].set_title('Calibration Plot')
plt.tight_layout(); plt.savefig('../data/processed/risk_model_eval.png', dpi=100, bbox_inches='tight'); plt.show()

In [ ]:
import joblib, os
os.makedirs('../api/models', exist_ok=True)
joblib.dump(model, '../api/models/risk_model.pkl')
joblib.dump(list(X.columns), '../api/models/feature_cols.pkl')
print("Model saved to api/models/risk_model.pkl")